# 02 (v2) - Prepare DeepPCB -> YOLO with a BOARD-level split
Runtime: **CPU**. Writes `prepared_v2.tar`, `split_manifest_v2.csv`, `manifest_v2.sha256`.

**What was wrong in v1:** DeepPCB's official train/test split is by *image* inside the same 11 boards, so every board landed in "test" and nothing was left to train on. v2 ignores the official lists and splits the **11 boards** itself: 6 train / 2 val / 3 test (fixed seed), plus a `fold` column (0-3) for leave-boards-out cross-validation.

v1 files are moved to `99_archive/wrong_split_v1/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import sys, os
D = '/content/drive/MyDrive/pcbrisk'
sys.path.insert(0, f'{D}/code')
from pcbrisk_utils import *
seed_all(42)

Mounted at /content/drive


In [ ]:
# archive the broken v1 artifacts (they would otherwise be picked up / block v2)
import shutil
arch = f'{D}/99_archive/wrong_split_v1'; os.makedirs(arch, exist_ok=True)
old = [f'{D}/01_prepared/{x}' for x in ['prepared_v1.tar','split_manifest_v1.csv','class_map.yaml','manifest_v1.sha256']] + \
      [f'{D}/02_synth/{x}' for x in ['synth_v1.tar','synth_meta_v1.csv','synth_v1_config.json']]
for f in old:
    if os.path.exists(f):
        shutil.move(f, f'{arch}/{os.path.basename(f)}'); print('archived', f)

archived /content/drive/MyDrive/pcbrisk/01_prepared/prepared_v1.tar
archived /content/drive/MyDrive/pcbrisk/01_prepared/split_manifest_v1.csv
archived /content/drive/MyDrive/pcbrisk/01_prepared/class_map.yaml
archived /content/drive/MyDrive/pcbrisk/01_prepared/manifest_v1.sha256
archived /content/drive/MyDrive/pcbrisk/02_synth/synth_v1.tar
archived /content/drive/MyDrive/pcbrisk/02_synth/synth_meta_v1.csv
archived /content/drive/MyDrive/pcbrisk/02_synth/synth_v1_config.json


In [ ]:
import glob, re, numpy as np, pandas as pd, yaml
from pathlib import Path

if not os.path.exists(f'{L}/raw/DeepPCB'):
    sh(f'unzip -q {D}/00_raw/deeppcb_main.zip -d {L}/raw')
P = Path(f'{L}/raw/DeepPCB/PCBData')
NAMES = ['open','short','mousebite','spur','copper','pin-hole']

boards = sorted(p.name for p in P.glob('group*'))
assert len(boards) == 11, boards
order = [str(b) for b in np.random.default_rng(42).permutation(boards)]
test_b, val_b, train_b = set(order[:3]), set(order[3:5]), set(order[5:])
fold_of = {b: i % 4 for i, b in enumerate(order)}          # for leave-boards-out CV
print('train', sorted(train_b)); print('val', sorted(val_b)); print('test', sorted(test_b))

train ['group12000', 'group12100', 'group13000', 'group20085', 'group77000', 'group90100']
val ['group12300', 'group92000']
test ['group00041', 'group44000', 'group50600']


In [ ]:
out = Path(f'{L}/data/yolo_all'); shutil.rmtree(out, ignore_errors=True)
(out/'images').mkdir(parents=True); (out/'labels').mkdir(parents=True)
rows, no_ann = [], 0
for gdir in sorted(P.glob('group*')):
    b = gdir.name
    split = 'test' if b in test_b else ('val' if b in val_b else 'train')
    for img in sorted(gdir.glob('*/*_test.jpg')):
        stem = img.name.replace('_test.jpg', '')
        ann = gdir / (img.parent.name + '_not') / f'{stem}.txt'
        shutil.copy2(img, out/'images'/f'{stem}.jpg')
        lines, counts = [], [0]*6
        if ann.exists():
            for l in open(ann):
                v = re.split(r'[,\s]+', l.strip())
                if len(v) < 5: continue
                x1, y1, x2, y2, c = map(float, v[:5]); c = int(c) - 1
                if c < 0 or c > 5: continue
                counts[c] += 1
                lines.append(f'{c} {(x1+x2)/2/640:.6f} {(y1+y2)/2/640:.6f} {(x2-x1)/640:.6f} {(y2-y1)/640:.6f}')
        else:
            no_ann += 1
        open(out/'labels'/f'{stem}.txt', 'w').write('\n'.join(lines))
        rows.append(dict(image_id=stem, source_board=b, split=split, fold=fold_of[b], n_defects=len(lines),
                         **{f'c{i}': counts[i] for i in range(6)}))
man = pd.DataFrame(rows); man['image_id'] = man.image_id.astype(str)
print(len(man), 'images; missing annotation files:', no_ann)

1500 images; missing annotation files: 0


In [ ]:
# ---- hard checks ----
assert man.image_id.is_unique
assert man.groupby('source_board').split.nunique().max() == 1, 'board in >1 split'
print(man.groupby('split').agg(images=('image_id','count'), boards=('source_board','nunique')))
cc = [f'c{i}' for i in range(6)]
tab = man.groupby('split')[cc].sum(); tab.columns = NAMES; print(tab)
assert (tab > 0).all().all(), 'a class is missing in a split - change the seed in default_rng(42)'
print(man.groupby('fold').source_board.nunique().to_dict(), '<- boards per CV fold')

temps = {p.name.replace('_temp.jpg','') for p in P.glob('group*/*/*_temp.jpg')}
print('templates without a test image:', temps - set(man.image_id))

       images  boards
split                
test      400       3
train     882       6
val       218       2
       open  short  mousebite  spur  copper  pin-hole
split                                                
test    549    402        591   456     401       422
train  1118    929       1159   966     889       871
val     275    175        215   203     184       208
{0: 3, 1: 3, 2: 3, 3: 2} <- boards per CV fold
templates without a test image: {'90100034'}


In [ ]:
yaml.safe_dump({'names': {i: n for i, n in enumerate(NAMES)}}, open(f'{L}/data/class_map.yaml', 'w'))
man_path = f'{L}/data/split_manifest_v2.csv'; man.to_csv(man_path, index=False)

if os.path.exists(f'{D}/01_prepared/prepared_v2.tar'):
    print('prepared_v2 already exists - immutable. Bump to v3 to change.')
else:
    sh(f'cd {L}/data && tar -cf {L}/prepared_v2.tar yolo_all class_map.yaml')
    safe_copy(f'{L}/prepared_v2.tar', f'{D}/01_prepared/prepared_v2.tar')
    safe_copy(man_path, f'{D}/01_prepared/split_manifest_v2.csv')
    safe_copy(f'{L}/data/class_map.yaml', f'{D}/01_prepared/class_map.yaml')
    open(f'{D}/01_prepared/manifest_v2.sha256', 'w').write(sha256_file(man_path))
    print('saved; manifest hash', sha256_file(man_path)[:12])

# round-trip check: ids must survive a CSV reload (leading zeros!)
chk = pd.read_csv(f'{D}/01_prepared/split_manifest_v2.csv', dtype={'image_id': str})
assert set(chk.image_id) == set(man.image_id)

saved; manifest hash 1f5bffd29c0d


**Always** read the manifest with `pd.read_csv(..., dtype={'image_id': str})`; plain `read_csv` turns `00041000` into `41000`.